In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/voffmail2/zipzip/test/f038b9d1-1d18-4b41-84f6-d22699d90040.jsonl
/kaggle/input/datasets/voffmail2/zipzip/test/a42b5c02-b03b-4a1b-ac3e-c99e7a2e0f50.jsonl
/kaggle/input/datasets/voffmail2/zipzip/test/913b3b86-7378-4248-8407-431069bcf31f.jsonl
/kaggle/input/datasets/voffmail2/zipzip/test/fe9ede0e-b2c7-4342-b02c-7266cc9be6c2.jsonl
/kaggle/input/datasets/voffmail2/zipzip/test/d8fa4b70-9267-4c16-a9c4-2fc2762123f6.jsonl
/kaggle/input/datasets/voffmail2/zipzip/test/ebdc5ec6-8fdf-41e8-93ea-42cd16d009f0.jsonl
/kaggle/input/datasets/voffmail2/zipzip/test/e46efa0f-35c5-4aa2-9ef8-70064fb37fc0.jsonl
/kaggle/input/datasets/voffmail2/zipzip/test/817f3f3c-34d8-435b-9ed0-ad633793ef17.jsonl
/kaggle/input/datasets/voffmail2/zipzip/test/7edf4eef-384a-48dc-aa1b-c927e3614fcc.jsonl
/kaggle/input/datasets/voffmail2/zipzip/test/dccc7785-7210-42f8-879d-16df20c14ebd.jsonl
/kaggle/input/datasets/voffmail2/zipzip/test/ae13b478-8d63-4ae0-9e69-ba742981f7a5.jsonl
/kaggle/input/datasets/voffmail2

In [2]:
#Score: 0.94117
#Public score: 0.90342
# %% [code]
# Устанавливаем нужные версии
!pip install lightgbm==4.5.0 optuna==4.2.1 scikit-learn==1.6.1

# %% [code]
import pandas as pd
import numpy as np
import glob
import json
from tqdm import tqdm
from pathlib import Path
import re
from collections import Counter
import math
import warnings

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import precision_recall_curve, classification_report, f1_score

import lightgbm as lgb
import optuna

warnings.filterwarnings("ignore")

# %% [code]
class Config:
    BASE_PATH = '/kaggle/input/datasets/voffmail2/zipzip/'
    TRAIN_PATH = f"{BASE_PATH}/train"
    TEST_PATH = f"{BASE_PATH}/test"
    
    RANDOM_STATE = 42
    TEST_SIZE = 0.2
    
    CHAR_NGRAM_MIN = 3
    CHAR_NGRAM_MAX = 7
    CHAR_MAX_FEATURES = 5000
    
    WORD_MAX_FEATURES = 1000

# %% [code]
def shannon_entropy(s):
    """Calculates the Shannon entropy of a string."""
    if not s or len(s) == 0:
        return 0
    p, lns = Counter(s), float(len(s))
    return -sum(count / lns * math.log(count / lns, 2) for count in p.values())

def extract_features_from_line(json_line):
    """
    Parses a single JSONL line and extracts a dictionary of features.
    """
    try:
        obj = json.loads(json_line)
        if len(obj) < 3:
            return None
        
        data = obj[2]
        rqs = data.get("rqs", {})
        rsp = data.get("rsp", {})
        
        url = rqs.get("url", "")
        host = rqs.get("host", "")
        
        features = {
            "rqs_method": rqs.get("method", "missing"),
            "rqs_host": host,
            "rqs_user_agent": rqs.get("user-agent", rqs.get("user_agent", "missing")),
            "rqs_url": url,
            "rsp_code": str(rsp.get("code", "missing")),
            "rsp_content_type": rsp.get("content-type", "missing"),
        }
        
        # Numerical URL/Host features
        features["url_len"] = len(url)
        features["host_len"] = len(host)
        features["url_path_depth"] = url.count("/")
        features["url_query_params"] = url.count("?") + url.count("&")
        features["url_entropy"] = shannon_entropy(url)
        features["host_entropy"] = shannon_entropy(host)
        
        # Additional features
        features["url_special_chars"] = len(re.findall(r'[%$&+,\/:;=?@<>#%{}\[\]\\]', url))
        features["url_digits"] = sum(c.isdigit() for c in url)
        features["host_digits"] = sum(c.isdigit() for c in host)
        features["host_is_ip"] = int(bool(re.match(r"^\d{1,3}(\.\d{1,3}){3}$", host)))
        features["ua_len"] = len(features["rqs_user_agent"])
        
        return features
    
    except Exception as e:
        return None

# %% [code]
# Data Splitting
clean_files = glob.glob(f"{Config.TRAIN_PATH}/clean/*.jsonl")
malware_files = glob.glob(f"{Config.TRAIN_PATH}/malware/*.jsonl")

file_data = [(f, 0) for f in clean_files] + [(f, 1) for f in malware_files]
file_df = pd.DataFrame(file_data, columns=["filepath", "label"])

train_files_df, val_files_df = train_test_split(
    file_df,
    test_size=Config.TEST_SIZE,
    random_state=Config.RANDOM_STATE,
    stratify=file_df["label"],
)

print(f"Total files: {len(file_df)}")
print(f"Training files: {len(train_files_df)} (Clean: {len(train_files_df[train_files_df['label']==0])}, Malware: {len(train_files_df[train_files_df['label']==1])})")
print(f"Validation files: {len(val_files_df)} (Clean: {len(val_files_df[val_files_df['label']==0])}, Malware: {len(val_files_df[val_files_df['label']==1])})")

# %% [code]
def load_data_from_files(file_list_df):
    """
    Loads all transactions from a list of files into a DataFrame.
    """
    transaction_data = []
    for _, row in tqdm(file_list_df.iterrows(), total=len(file_list_df), desc="Loading data"):
        filepath = row["filepath"]
        label = row["label"]
        
        try:
            with open(filepath, "r", encoding="utf-8") as f:
                for line in f:
                    features = extract_features_from_line(line)
                    if features:
                        features["label"] = label
                        features["filepath"] = filepath
                        transaction_data.append(features)
        except Exception as e:
            print(f"Error loading {filepath}: {e}")
            continue
    
    return pd.DataFrame(transaction_data)

print("Loading training transactions...")
train_df = load_data_from_files(train_files_df)

y_train = train_df["label"]
X_train = train_df.drop(["label", "filepath"], axis=1)

print(f"\nLoaded {len(train_df)} transactions for training.")
print("Class distribution of transactions:")
print(y_train.value_counts(normalize=True))

# %% [code]
# Feature Preprocessing Pipeline
print("Building preprocessing pipeline...")

numerical_features = [
    'url_len', 'host_len', 'url_path_depth', 'url_query_params', 
    'url_entropy', 'host_entropy', 'url_special_chars', 'url_digits',
    'host_digits', 'ua_len'
]

categorical_features = ['rqs_method', 'rsp_code', 'host_is_ip']

numerical_pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler()
)

categorical_pipeline = make_pipeline(
    SimpleImputer(strategy='constant', fill_value='missing'),
    OneHotEncoder(handle_unknown='ignore', sparse_output=True)
)

# Char-level TF-IDF на символьных n-граммах
text_char_pipeline = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(Config.CHAR_NGRAM_MIN, Config.CHAR_NGRAM_MAX),
    max_features=Config.CHAR_MAX_FEATURES,
    sublinear_tf=True,
    use_idf=True,
    smooth_idf=True
)

text_word_pipeline = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),
    max_features=Config.WORD_MAX_FEATURES,
    sublinear_tf=True,
    use_idf=True,
    smooth_idf=True
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_pipeline, numerical_features),
        ('cat', categorical_pipeline, categorical_features),
        ('tfidf_char_url', text_char_pipeline, 'rqs_url'),
        ('tfidf_char_host', text_char_pipeline, 'rqs_host'),
        ('tfidf_word_ua', text_word_pipeline, 'rqs_user_agent'),
        ('tfidf_word_ct', text_word_pipeline, 'rsp_content_type'),
    ],
    remainder='drop',
    n_jobs=-1
)

print("Pipeline built successfully.")

# %% [code]
print("Fitting preprocessing pipeline on X_train...")
X_train_transformed = preprocessor.fit_transform(X_train)
print(f"Training data transformed. Shape: {X_train_transformed.shape}")

class_counts = y_train.value_counts()
scale_pos_weight = class_counts[0] / class_counts[1]
print(f"Calculated scale_pos_weight: {scale_pos_weight:.2f}")

# %% [code]
# Оптимизация гиперпараметров с помощью Optuna
def objective(trial):
    params = {
        'random_state': Config.RANDOM_STATE,
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'scale_pos_weight': scale_pos_weight,
        'n_jobs': -1,
        'verbosity': -1
    }
    
    # Кросс-валидация на 3 фолдах
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=Config.RANDOM_STATE)
    cv_scores = []
    
    for train_idx, val_idx in skf.split(X_train_transformed, y_train):
        X_tr = X_train_transformed[train_idx]
        X_va = X_train_transformed[val_idx]
        y_tr = y_train.iloc[train_idx]
        y_va = y_train.iloc[val_idx]
        
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
        )
        
        preds = model.predict_proba(X_va)[:, 1]
        preds_binary = (preds >= 0.5).astype(int)
        cv_scores.append(f1_score(y_va, preds_binary))
    
    return np.mean(cv_scores)

print("Running hyperparameter optimization...")
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=Config.RANDOM_STATE))
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f"\nBest parameters: {study.best_params}")
print(f"Best CV F1: {study.best_value:.4f}")

# %% [code]
# Train final model with best parameters
best_params = study.best_params
best_params.update({
    'random_state': Config.RANDOM_STATE,
    'scale_pos_weight': scale_pos_weight,
    'n_jobs': -1,
    'verbosity': -1
})

model = lgb.LGBMClassifier(**best_params)
model.fit(X_train_transformed, y_train)

# %% [code]
def predict_file_max_proba(filepath, preprocessor, model):
    """
    Loads, parses, transforms, and predicts probabilities for all transactions
    in a single file. Returns the MAX probability.
    """
    transactions_data = []
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                features = extract_features_from_line(line)
                if features:
                    transactions_data.append(features)
    except Exception as e:
        print(f"Error processing {filepath}: {e}")
        return 0.0
    
    if not transactions_data:
        return 0.0
    
    file_df = pd.DataFrame(transactions_data)
    file_transformed = preprocessor.transform(file_df)
    
    probs = model.predict_proba(file_transformed)[:, 1]
    return np.max(probs)

print("Evaluating model on validation files...")

val_scores = []
val_true_labels = []

for _, row in tqdm(val_files_df.iterrows(), total=len(val_files_df), desc="Validating"):
    filepath = row["filepath"]
    true_label = row["label"]
    
    max_proba = predict_file_max_proba(filepath, preprocessor, model)
    
    val_scores.append(max_proba)
    val_true_labels.append(true_label)

print("Validation complete.")

# %% [code]
# Find the best threshold
precision, recall, thresholds = precision_recall_curve(val_true_labels, val_scores)
f1_scores = (2 * precision * recall) / (precision + recall + 1e-9)

# Find the best (excluding last element which is 0)
best_f1_idx = np.argmax(f1_scores[:-1])
best_f1 = f1_scores[best_f1_idx]
best_threshold = thresholds[best_f1_idx]

print(f"\nValidation Results")
print(f"Best File-Level F1 Score: {best_f1:.4f}")
print(f"Best Probability Threshold: {best_threshold:.4f}")

# Print classification report at the best threshold
val_preds = (np.array(val_scores) >= best_threshold).astype(int)
print("\nClassification Report at Best Threshold:")
print(classification_report(val_true_labels, val_preds, target_names=["Clean (0)", "Malware (1)"]))

# %% [code]
print("\nGenerating test predictions...")
test_files = glob.glob(f"{Config.TEST_PATH}/*.jsonl")
submission = []

for filepath in tqdm(test_files, desc="Predicting on test set"):
    file_id = Path(filepath).stem
    file_max_proba = predict_file_max_proba(filepath, preprocessor, model)
    prediction = 1 if file_max_proba >= best_threshold else 0
    submission.append({'id': file_id, 'target': prediction})

submission_df = pd.DataFrame(submission)
submission_df.to_csv('submission.csv', index=False)

print("\nSubmission file created successfully!")
print(submission_df.head())
print("\nPredicted class distribution in test set:")
print(submission_df['target'].value_counts(normalize=True))

# %% [code]
print(f"\n=== FINAL RESULTS ===")
print(f"Validation F1: {best_f1:.4f}")
print(f"Threshold: {best_threshold:.4f}")
print(f"Test predictions - Malware: {submission_df['target'].sum()} out of {len(submission_df)} ({submission_df['target'].mean()*100:.1f}%)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 18.3 MB/s eta 0:00:00
  Attempting uninstall: lightgbm
    Found existing installation: lightgbm 4.6.0
    Uninstalling lightgbm-4.6.0:
      Successfully uninstalled lightgbm-4.6.0
  Attempting uninstall: optuna
    Found existing installation: optuna 4.7.0
    Uninstalling optuna-4.7.0:
      Successfully uninstalled optuna-4.7.0
Total files: 4435
Training files: 3548 (Clean: 2735, Malware: 813)
Validation files: 887 (Clean: 684, Malware: 203)
Loading training transactions...


Loading data: 100%|██████████| 3548/3548 [00:19<00:00, 177.56it/s]



Loaded 10424 transactions for training.
Class distribution of transactions:
label
0    0.604566
1    0.395434
Name: proportion, dtype: float64
Building preprocessing pipeline...
Pipeline built successfully.
Fitting preprocessing pipeline on X_train...


[I 2026-03-16 12:02:13,375] A new study created in memory with name: no-name-1908a07e-1baa-4d86-afaa-5a0cecb0ccb7


Training data transformed. Shape: (10424, 11262)
Calculated scale_pos_weight: 1.53
Running hyperparameter optimization...


...

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[76]	valid_0's binary_logloss: 0.234719
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[82]	valid_0's binary_logloss: 0.229615
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[52]	valid_0's binary_logloss: 0.241467
[I 2026-03-16 12:02:51,618] Trial 0 finished with value: 0.8927133253222107 and parameters: {'n_estimators': 250, 'learning_rate': 0.17254716573280354, 'num_leaves': 97, 'max_depth': 11, 'min_child_samples': 12, 'subsample': 0.662397808134481, 'colsample_bytree': 0.6232334448672797, 'reg_alpha': 0.8661761457749352, 'reg_lambda': 0.6011150117432088}. Best is trial 0 with value: 0.8927133253222107.
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[383]	valid_0's binary_logloss: 0.238398
Training until validation scores don't imp

Validating: 100%|██████████| 887/887 [00:49<00:00, 18.06it/s]


Validation complete.

Validation Results
Best File-Level F1 Score: 0.9363
Best Probability Threshold: 0.7139

Classification Report at Best Threshold:
              precision    recall  f1-score   support

   Clean (0)       0.98      0.98      0.98       684
 Malware (1)       0.93      0.94      0.94       203

    accuracy                           0.97       887
   macro avg       0.96      0.96      0.96       887
weighted avg       0.97      0.97      0.97       887


Generating test predictions...


Predicting on test set: 100%|██████████| 1326/1326 [01:11<00:00, 18.44it/s]


Submission file created successfully!
                                     id  target
0  f038b9d1-1d18-4b41-84f6-d22699d90040       1
1  a42b5c02-b03b-4a1b-ac3e-c99e7a2e0f50       0
2  913b3b86-7378-4248-8407-431069bcf31f       0
3  fe9ede0e-b2c7-4342-b02c-7266cc9be6c2       0
4  d8fa4b70-9267-4c16-a9c4-2fc2762123f6       0

Predicted class distribution in test set:
target
0    0.75641
1    0.24359
Name: proportion, dtype: float64

=== FINAL RESULTS ===
Validation F1: 0.9363
Threshold: 0.7139
Test predictions - Malware: 323 out of 1326 (24.4%)
